In [1]:
import sys
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "utils.py").exists() and (ROOT.parent / "src" / "utils.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import DATA_CLEANED, load_jsonl
from src.sentiment.ollama_scorer import _parse_sentiment_number, SENTIMENT_PROMPT, OLLAMA_URL

In [2]:
path = DATA_CLEANED / "processed_master_orig.jsonl"
rows = load_jsonl(path)
df = pd.DataFrame(rows)
df.head()

,posted_at,fetched_at,headline,url,source,reporter,ticker,is_ai_related,is_proxy_partnership,sentiment_vader,sentiment_llm_phi3,sentiment_llm_llama3_2,sentiment_llm_deepseek_r1
0,2026-02-23T21:47:00Z,2026-02-26T15:43:31Z,AirPods as Apple’s first AI wearable product m...,https://9to5mac.com/2026/02/23/airpods-as-appl...,NewsAPI Tech,9to5Mac,AAPL,True,False,0.0,0.8,0.4,0.7
1,2026-02-24T05:45:00Z,2026-02-26T15:43:31Z,Gemini 3.1 Pro is a powerhouse for deep work —...,https://www.tomsguide.com/ai/gemini-3-1-pro-is...,NewsAPI Tech,Tom's Guide,GOOGL,True,False,0.0,0.6,0.4,0.0
2,2026-02-24T16:57:21Z,2026-02-26T15:43:31Z,Music generator ProducerAI joins Google Labs -...,https://techcrunch.com/2026/02/24/music-genera...,NewsAPI Tech,TechCrunch,AAPL,True,True,0.0,0.6,0.4,0.9
3,2026-02-24T16:57:21Z,2026-02-26T15:43:31Z,Music generator ProducerAI joins Google Labs -...,https://techcrunch.com/2026/02/24/music-genera...,NewsAPI Tech,TechCrunch,GOOGL,True,False,0.0,0.6,0.4,0.9
4,2026-02-24T17:30:10Z,2026-02-26T15:43:31Z,Microsoft adds Copilot data controls to all st...,https://www.bleepingcomputer.com/news/microsof...,NewsAPI Tech,BleepingComputer,MSFT,True,False,0.0,0.6,-0.4,0.5


In [3]:
nan_deepseek = df[df["sentiment_llm_deepseek_r1"].isna()]
print(f"NaN deepseek rows: {len(nan_deepseek)}")

nan_deepseek[["posted_at", "ticker", "headline",
              "sentiment_vader", "sentiment_llm_phi3", "sentiment_llm_llama3_2",
              "sentiment_llm_deepseek_r1"]].head(20)

NaN deepseek rows: 94


,posted_at,ticker,headline,sentiment_vader,sentiment_llm_phi3,sentiment_llm_llama3_2,sentiment_llm_deepseek_r1
8,2026-02-24T20:40:00Z,MSFT,Nvidia Just Sold Its Stake in Applied Digital ...,0.4767,0.6,0.8,NaN
9,2026-02-24T20:40:00Z,NVDA,Nvidia Just Sold Its Stake in Applied Digital ...,0.4767,0.6,0.8,NaN
14,2026-02-24T23:01:28Z,AAPL,Apple's 2026 MacBook Pro Refresh Brings Dynami...,0.3818,0.6,0.4,NaN
27,2026-02-25T19:15:00Z,AAPL,Canada’s AI minister blames OpenAI for ‘failur...,-0.4019,-0.9,-0.9,NaN
28,2026-02-25T19:15:00Z,MSFT,Canada’s AI minister blames OpenAI for ‘failur...,-0.4019,-0.9,-0.9,NaN
29,2026-02-25T19:15:00Z,NVDA,Canada’s AI minister blames OpenAI for ‘failur...,-0.4019,-0.9,-0.9,NaN
64,2026-02-26T11:44:56Z,TSLA,Elon Musk's makeshift AI power plant generates...,-0.5719,-0.2,0.2,NaN
73,2026-02-26T16:00:00Z,AAPL,Google launches Nano Banana 2 model with faste...,0.0000,0.6,0.4,NaN
74,2026-02-26T16:00:00Z,GOOGL,Google launches Nano Banana 2 model with faste...,0.0000,0.6,0.4,NaN
75,2026-02-26T16:08:55Z,AAPL,Google rolls out Nano Banana 2 after viral suc...,0.5719,0.6,0.4,NaN


In [4]:
test_headlines = list(nan_deepseek["headline"].drop_duplicates().head(10))
test_headlines

['Nvidia Just Sold Its Stake in Applied Digital and Arm Holdings and Piled Into a New Artificial Intelligence GPU Player Up Over 7,200% Since Its IPO - Yahoo Finance',
 "Apple's 2026 MacBook Pro Refresh Brings Dynamic Island, OLED Screens, and New Touch Gestures - TechPowerUp",
 'Canada’s AI minister blames OpenAI for ‘failure’ after mass shooting - Politico',
 "Elon Musk's makeshift AI power plant generates sound and fury in Mississippi - NBC News",
 'Google launches Nano Banana 2 model with faster image generation - TechCrunch',
 'Google rolls out Nano Banana 2 after viral success of AI image generation tool - Yahoo Finance',
 'Read AI launches an email-based ‘digital twin’ to help you with schedules and answers',
 'So, we’re getting Prada Meta AI glasses, right?',
 'OpenAI raises $110B in one of the largest private funding rounds in history',
 'Supercharge your AI agents: The New ADK Integrations Ecosystem - blog.google']

In [5]:
def call_deepseek_raw(headline: str, context: str | None = None, model: str = "deepseek-r1:1.5b"):
    context_block = f"Context: {context}\n\n" if (context and context.strip()) else ""
    prompt = SENTIMENT_PROMPT.format(CONTEXT=context_block, HEADLINE=headline.strip())
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0},
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    return data.get("response", "")

In [6]:
for h in test_headlines:
    print("=" * 80)
    print("HEADLINE:", h)
    raw = call_deepseek_raw(h)
    parsed = _parse_sentiment_number(raw)
    print("RAW RESPONSE:")
    print(raw)
    print("PARSED NUMBER:", parsed)

HEADLINE: Nvidia Just Sold Its Stake in Applied Digital and Arm Holdings and Piled Into a New Artificial Intelligence GPU Player Up Over 7,200% Since Its IPO - Yahoo Finance
RAW RESPONSE:
-0.5
PARSED NUMBER: -0.5
HEADLINE: Apple's 2026 MacBook Pro Refresh Brings Dynamic Island, OLED Screens, and New Touch Gestures - TechPowerUp
RAW RESPONSE:
0.9
PARSED NUMBER: 0.9
HEADLINE: Canada’s AI minister blames OpenAI for ‘failure’ after mass shooting - Politico
RAW RESPONSE:
-0.5
PARSED NUMBER: -0.5
HEADLINE: Elon Musk's makeshift AI power plant generates sound and fury in Mississippi - NBC News
RAW RESPONSE:
0.4
PARSED NUMBER: 0.4
HEADLINE: Google launches Nano Banana 2 model with faster image generation - TechCrunch


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)

ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)

maybe increasing time out time will help
